In [74]:
import torch
from einops import rearrange
import numpy as np

In [12]:
raw_rmse = torch.ones((2, 4, 1, 1)) # (B, C, 1, 1)
gamma = torch.ones((2, 4, 6, 8)) # (B, C, H, W)

In [19]:
def get_curved_scores(raw_rmse: torch.Tensor, gamma: float) -> torch.Tensor:
    # gamma: (B, C, H, W)
    min_rmse = rearrange(torch.tensor((0.1, 0.0, 0.4, 0.4)), 'c -> c 1 1')    # Mike!!! Rough. They come from w&d
    max_rmse = raw_rmse * 1.2   # Mike!!! A rouph assumption

    min_rmse = torch.minimum(min_rmse, raw_rmse)
    eq = (min_rmse == max_rmse)

    amin = torch.minimum(min_rmse, max_rmse)
    amax = torch.maximum(min_rmse, max_rmse)
    den  = (amax - amin)
    norm = (raw_rmse - amin) / torch.clamp(den, min=1e-12)
    norm = torch.where(eq, torch.ones_like(norm), norm)

    return torch.pow(norm, gamma)  # gamma.shape = (B, C, H, W)

In [23]:
def get_rmse(target: torch.Tensor, prediction: torch.Tensor, dims: tuple) -> torch.Tensor:
    try:
        return ((prediction - target) ** 2).mean(dim=dims).sqrt()
    except:
        return - torch.ones_like(prediction.mean(dim=dims))

In [ ]:
# def loss(self, target, pred, batch_diff, era5_data_vars):
#     # (B,FG,C,H,W), (B,FG,C,H,W), (B, C, H, W), (C,)
target = torch.rand((2, 24, 4, 600, 800)) - 0.5
pred = torch.rand((2, 24, 4, 600, 800)) - 0.5
min_sota_delta = torch.rand((4,))
batch_diff = torch.rand((2, 4, 600, 800))
era5_data_vars = torch.rand((4,))

baseline_rmse = torch.tensor(0.0)               # (,)

rmse = get_rmse(target, pred, dims=(1, 3, 4))    # (B, C)  the-higher-the-worse
improvement = baseline_rmse - rmse  - min_sota_delta   # (B, C)  the-higher-the-better
baseline_improvement = torch.maximum(torch.tensor(0.0), improvement)    # 0.0, (B, C)  the-higher-the-better

gamma = np.power(3.0, batch_diff * 2 - 1)  # (B, C, H, W)
quality_score = get_curved_scores(rearrange(rmse, 'b c -> b c 1 1' ), gamma)       # (B, C, H, W) the-higher-the-worse
baseline_improvement = rearrange(baseline_improvement, 'b c -> b c 1 1')                # 0.0, (B, C, 1, 1)
improvement_score = get_curved_scores(-baseline_improvement, gamma) # 1.0, (B, C, H, W) the-higher-the-worse
reward = (1 - 0.5) * quality_score + 0.5 * improvement_score    # (B, C, H, W)
reward = reward * rearrange(era5_data_vars, 'c -> c 1 1')   # (B, 1, C, H, W) the-higher-the-worse
reward = reward.mean()

# print(reward)
# return reward

tensor([[[[[ 4.6557e-01,  3.2271e-01, -6.8159e-02,  ..., -1.2529e-01,
            -2.5789e-01,  4.0557e-01],
           [-2.6048e-01, -4.9714e-01, -3.1574e-01,  ...,  4.0255e-01,
            -2.2085e-01,  2.9713e-01],
           [ 4.1931e-02,  3.3658e-01,  9.9505e-02,  ...,  3.2894e-01,
            -4.5355e-01, -2.2619e-01],
           ...,
           [ 3.1771e-01,  1.2523e-02,  4.0593e-02,  ..., -1.3560e-01,
             4.9415e-01,  2.2311e-01],
           [ 3.3821e-01, -4.9504e-01, -1.5452e-01,  ...,  2.9992e-01,
            -2.1210e-01,  2.4857e-01],
           [ 9.7872e-02,  2.5051e-02, -3.0463e-01,  ..., -3.4192e-02,
            -2.1390e-01, -1.4242e-01]],

          [[ 5.1860e-02, -4.9080e-01, -4.5045e-01,  ..., -4.8866e-01,
             3.2746e-01,  4.2436e-01],
           [ 3.5602e-01,  4.8816e-01,  4.5002e-02,  ..., -8.6493e-03,
            -2.0695e-01,  4.0452e-02],
           [-1.2916e-01, -3.0189e-01, -7.9790e-02,  ..., -3.5253e-01,
            -3.3589e-01, -1.3240e-01],
 

In [2]:
import sys
import os
import numpy as np
import torch
from torch import nn
from tqdm import tqdm

from einops.layers.torch import Rearrange
from einops import rearrange, repeat
from fvcore.nn import FlopCountAnalysis, flop_count_table


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [9]:
sys.path.append(os.path.abspath(os.path.join('..', 'models')))

from Zeus_Trainer import Trainer

In [ ]:
def sample_time_range(self) -> Tuple[pd.Timestamp, pd.Timestamp, int]:
    """
    Sample random start and end times according to the provided ranges.
    """
    num_predict_hours = np.random.randint(*self.predict_sample_range)

        # see visualisation at Zeus/static/era5_start_offset_distribution.png
    if np.random.rand() > self.uniform_start_prob:
        start_offset = min(
            self.start_offset_range[1], # don't overshoot
            np.abs(
                int(np.random.normal(0, self.start_sample_std))
            ) + self.start_offset_range[0]
        )
    else:
        start_offset = int(np.random.uniform(*self.start_offset_range))

    start_timestamp = get_today("h") + pd.Timedelta(hours=start_offset)
    end_timestamp = start_timestamp + pd.Timedelta(hours=num_predict_hours - 1)

    return start_timestamp, end_timestamp, num_predict_hours

In [ ]:
def get_curved_scores(raw_scores: List[float], gamma: float) -> List[float]:
    """
    Given a list of raw float scores (can by any range),
    normalise them to 0-1 scores,
    and apply gamma correction to curve accordingly.

    This function assumes lower is better
    """
    min_score = min(raw_scores)
    max_score = max(raw_scores)

    result = []
    for score in raw_scores:
        if max_score == min_score:
            result.append(1.0) # edge case, avoid division by 0
        else:
            norm_rmse = (max_score - score) / (max_score - min_score)
            result.append(np.power(norm_rmse, gamma)) # apply gamma correction
    
    return result

In [ ]:
def set_rewards(
    output_data: torch.Tensor,
    miners_data: List[MinerData],
    baseline_data: Optional[torch.Tensor],
    difficulty_grid: np.ndarray,
    min_sota_delta: float
) -> List[MinerData]:
    """
    Calculates rewards for miner predictions based on RMSE and relative difficulty.
    NOTE: it is assumed penalties have already been scored and filtered out, 
      if not will do so without scoring those

    Args:
        output_data (torch.Tensor): The ground truth data.
        miners_data (List[MinerData]): List of MinerData objects containing predictions.
        baseline_data (torch.Tensor): OpenMeteo prediction, where additional incentive is awarded to beat this!
        difficulty_grid (np.ndarray): Difficulty grid for each coordinate.

    Returns:
        List[MinerData]: List of MinerData objects with updated rewards and metrics.
    """
    miners_data = [m for m in miners_data if not m.shape_penalty]

    if len(miners_data) == 0:
        return miners_data

    # old challenges have no baseline, use 0 to make it not affect scoring.
    baseline_rmse = 0
    if baseline_data is not None:
        baseline_rmse = rmse(output_data, baseline_data)
        
    avg_difficulty = difficulty_grid.mean()
    # make difficulty [-1, 1], then go between [1/scaler, scaler]
    gamma = np.power(REWARD_DIFFICULTY_SCALER, avg_difficulty * 2 - 1)

    # compute unnormalised scores
    for miner_data in miners_data:
        miner_data.rmse = rmse(output_data, miner_data.prediction)
        improvement = baseline_rmse - miner_data.rmse - min_sota_delta
        miner_data.baseline_improvement = max(0, improvement)

    quality_scores = get_curved_scores([m.rmse for m in miners_data], gamma)
    # negative since curving assumes minimal is the best
    improvement_scores = get_curved_scores([-m.baseline_improvement for m in miners_data], gamma)

    for miner_data, quality, improvement in zip(miners_data, quality_scores, improvement_scores):
        miner_data.reward = (1 - REWARD_IMPROVEMENT_WEIGHT) * quality + REWARD_IMPROVEMENT_WEIGHT * improvement

    return miners_data

In [ ]:
config = {
    # image h w c
    'height': 72,
    'width': 144,
    'channels': 2,
    # video length in and out
    'T1': 1,    # 5 later.
    'MaxT2': 5,
    'FG': 24,   # frames in a frame group
    # patch size
    'patch_size': 8,
    # transformer
    'heads': 8,
    'dim_head': 32,
    'dropout': 0.0,
    'attn_dropout': 0.0,
    'drop_path': 0.0,
    'patch_dim_to_dim' : 1,
    'dim_to_mlp_dim': 2,
    # layers
    'attLayers' : 1,
    'zeusLayers': 6,
}

In [4]:
trainer = Trainer(config)

In [5]:
x = torch.rand(1, 24, 2, 72, 144)
output = trainer.model(x)
print(output.shape)  # [B, T, C, H, W]
# # Calculate FLOPs
flops = FlopCountAnalysis(trainer.model, x)
print(f'Number of flops: {flop_count_table(flops)}')


torch.Size([1, 24, 2, 72, 144])
Number of flops: | module                              | #parameters or shape   | #flops    |
|:------------------------------------|:-----------------------|:----------|
| model                               | 7.947M                 | 35.341G   |
|  to_patch_embedding.1               |  16.512K               |  63.701M  |
|   to_patch_embedding.1.weight       |   (128, 128)           |           |
|   to_patch_embedding.1.bias         |   (128,)               |           |
|  blocks                             |  7.913M                |  35.199G  |
|   blocks.0                          |   1.319M               |   5.866G  |
|    blocks.0.ts_temporal_transformer |    0.33M               |    1.329G |
|    blocks.0.ts_space_transformer    |    0.33M               |    1.604G |
|    blocks.0.st_space_transformer    |    0.33M               |    1.604G |
|    blocks.0.st_temporal_transformer |    0.33M               |    1.329G |
|   blocks.1               